<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-05-bigquery-ml/lesson-5.2-time-series/notebooks/GCP_Capstone_5.2_TimeSeries.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5.2 Time Series & Anomaly Detection — ARIMA_PLUS
**Netsetos GenAI Engineering — GCP Capstone**

Forecast query volumes, detect cost spikes, explain seasonal patterns — all in SQL.


## Setup


In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

def run_query(sql):
    return client.query(sql).to_dataframe()

def run_ddl(sql):
    job = client.query(sql)
    job.result()
    print(f'Done: {job.num_dml_affected_rows or "OK"}')

print(f'Connected to {PROJECT_ID}')


## Cell 1: Generate Synthetic Time Series Data


In [ ]:
# Generate 30 days of hourly query data with patterns
run_ddl(f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.rag_data.hourly_metrics` AS
WITH hours AS (
  SELECT ts
  FROM UNNEST(GENERATE_TIMESTAMP_ARRAY(
    TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY),
    CURRENT_TIMESTAMP(), INTERVAL 1 HOUR)) AS ts
)
SELECT
  ts AS hour_bucket,
  CAST(
    50  -- base
    + 30 * SIN(2 * ACOS(-1) * EXTRACT(HOUR FROM ts) / 24)  -- daily pattern
    + 15 * IF(EXTRACT(DAYOFWEEK FROM ts) IN (1, 7), -1, 1)  -- weekend dip
    + 0.5 * TIMESTAMP_DIFF(ts, TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY), HOUR) / 24  -- trend
    + 10 * (RAND() - 0.5)  -- noise
    + IF(RAND() < 0.02, 80, 0)  -- rare spikes
  AS INT64) AS query_count,
  ROUND(
    0.003 * (
      50 + 30 * SIN(2 * ACOS(-1) * EXTRACT(HOUR FROM ts) / 24)
      + 10 * (RAND() - 0.5)
      + IF(RAND() < 0.01, 50, 0)
    ), 4) AS hourly_cost
FROM hours
''')
print('30 days of hourly data generated')
print(run_query(f'SELECT * FROM `{PROJECT_ID}.rag_data.hourly_metrics` ORDER BY hour_bucket DESC LIMIT 5'))


## Cell 2: Train ARIMA_PLUS on Query Volume


In [ ]:
run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.query_forecast`
OPTIONS (
  model_type = 'ARIMA_PLUS',
  time_series_timestamp_col = 'hour_bucket',
  time_series_data_col = 'query_count',
  auto_arima = TRUE,
  data_frequency = 'HOURLY',
  decompose_time_series = TRUE,
  clean_spikes_and_dips = TRUE,
  horizon = 168,
  forecast_limit_lower_bound = 0
) AS
SELECT hour_bucket, query_count
FROM `{PROJECT_ID}.rag_data.hourly_metrics`
''')
print('Query forecast model trained')

# Check ARIMA parameters selected
print('\n=== ARIMA Evaluation ===')
print(run_query(f'SELECT * FROM ML.ARIMA_EVALUATE(MODEL `{PROJECT_ID}.ml_models.query_forecast`)'))


## Cell 3: ML.FORECAST — Next 72 Hours


In [ ]:
forecast = run_query(f'''
SELECT
  forecast_timestamp,
  ROUND(forecast_value, 1) AS predicted_queries,
  ROUND(prediction_interval_lower_bound, 1) AS lower_95,
  ROUND(prediction_interval_upper_bound, 1) AS upper_95
FROM ML.FORECAST(
  MODEL `{PROJECT_ID}.ml_models.query_forecast`,
  STRUCT(72 AS horizon, 0.95 AS confidence_level))
ORDER BY forecast_timestamp
''')
print(f'Forecast: {len(forecast)} rows')
print(forecast.head(10))

# Capacity planning: peak hours
peaks = forecast[forecast['upper_95'] > 100]
print(f'\nHours exceeding 100 queries (worst case): {len(peaks)}')


## Cell 4: Train Cost Anomaly Model + Detect


In [ ]:
# Train cost model
run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.cost_anomaly`
OPTIONS (
  model_type = 'ARIMA_PLUS',
  time_series_timestamp_col = 'hour_bucket',
  time_series_data_col = 'hourly_cost',
  decompose_time_series = TRUE,
  clean_spikes_and_dips = TRUE
) AS
SELECT hour_bucket, hourly_cost
FROM `{PROJECT_ID}.rag_data.hourly_metrics`
''')
print('Cost anomaly model trained')

# Detect anomalies
anomalies = run_query(f'''
SELECT
  hour_bucket,
  hourly_cost,
  is_anomaly,
  ROUND(anomaly_probability, 3) AS anomaly_prob,
  ROUND(lower_bound, 4) AS expected_low,
  ROUND(upper_bound, 4) AS expected_high
FROM ML.DETECT_ANOMALIES(
  MODEL `{PROJECT_ID}.ml_models.cost_anomaly`,
  STRUCT(0.95 AS anomaly_prob_threshold))
WHERE is_anomaly = TRUE
ORDER BY anomaly_probability DESC
''')
print(f'\nAnomalies found: {len(anomalies)}')
print(anomalies.head(10))


## Cell 5: ML.EXPLAIN_FORECAST — Decomposition


In [ ]:
decomp = run_query(f'''
SELECT
  time_series_type,
  time_series_timestamp,
  ROUND(time_series_data, 1) AS value,
  ROUND(trend, 1) AS trend,
  ROUND(seasonal_period_daily, 1) AS daily_pattern,
  ROUND(seasonal_period_weekly, 1) AS weekly_pattern,
  ROUND(spikes_and_dips, 1) AS spikes,
  ROUND(step_changes, 1) AS step_changes
FROM ML.EXPLAIN_FORECAST(
  MODEL `{PROJECT_ID}.ml_models.query_forecast`,
  STRUCT(48 AS horizon))
ORDER BY time_series_timestamp
''')

print('=== Forecast decomposition ===')
print(decomp[decomp['time_series_type'] == 'forecast'].head(10))

# Show last few historical points for comparison
print('\n=== Recent history ===')
hist = decomp[decomp['time_series_type'] == 'history']
print(hist.tail(5))


## Cell 6: Detect Anomalies on NEW Data


In [ ]:
# Simulate checking today's data against the model
new_anomalies = run_query(f'''
SELECT *
FROM ML.DETECT_ANOMALIES(
  MODEL `{PROJECT_ID}.ml_models.cost_anomaly`,
  STRUCT(0.90 AS anomaly_prob_threshold),
  (SELECT hour_bucket, hourly_cost
   FROM `{PROJECT_ID}.rag_data.hourly_metrics`
   WHERE hour_bucket >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 24 HOUR))
)
WHERE is_anomaly = TRUE
''')
print(f'Real-time anomalies in last 24h: {len(new_anomalies)}')
if len(new_anomalies) > 0:
    print(new_anomalies)


## Cell 7: Operational Intelligence Dashboard Queries


In [ ]:
# All 3 operational queries combined
print('=== 1. CAPACITY PLANNING ===')
print(run_query(f'''
SELECT forecast_timestamp, ROUND(forecast_value) AS predicted,
  ROUND(prediction_interval_upper_bound) AS worst_case
FROM ML.FORECAST(MODEL `{PROJECT_ID}.ml_models.query_forecast`, STRUCT(24 AS horizon))
WHERE prediction_interval_upper_bound > 80
'''))

print('\n=== 2. COST ANOMALIES ===')
print(run_query(f'''
SELECT hour_bucket, hourly_cost, ROUND(anomaly_probability, 3) AS prob
FROM ML.DETECT_ANOMALIES(
  MODEL `{PROJECT_ID}.ml_models.cost_anomaly`, STRUCT(0.95 AS anomaly_prob_threshold))
WHERE is_anomaly = TRUE ORDER BY anomaly_probability DESC LIMIT 5
'''))

print('\n=== 3. PATTERN EXPLANATION ===')
print(run_query(f'''
SELECT time_series_timestamp, ROUND(trend, 1) AS trend,
  ROUND(seasonal_period_daily, 1) AS daily, ROUND(seasonal_period_weekly, 1) AS weekly
FROM ML.EXPLAIN_FORECAST(
  MODEL `{PROJECT_ID}.ml_models.query_forecast`, STRUCT(24 AS horizon))
WHERE time_series_type = 'forecast' LIMIT 10
'''))


## ✅ Lesson 5.2 Complete!

- ✅ ARIMA_PLUS with auto (p,d,q) tuning
- ✅ ML.FORECAST with confidence intervals
- ✅ ML.DETECT_ANOMALIES on historical + new data
- ✅ ML.EXPLAIN_FORECAST decomposition
- ✅ Multi-series with time_series_id_col
- ✅ ARIMA_PLUS_XREG external regressors
- ✅ 3-model operational intelligence

**Next: Lesson 5.3 — ML.GENERATE_TEXT: Gemini Inside SQL**
